# Stable Diffusion 1.5 from Scratch

This tutorial walks through every component of **Stable Diffusion 1.5** (SD 1.5)
so you can understand how text-to-image and image-to-image generation actually work
under the hood.

## High-Level Architecture

SD 1.5 is a **Latent Diffusion Model (LDM)**. Instead of diffusing directly in
pixel space (which would be extremely expensive at 512×512×3), it operates in a
compressed **latent space** produced by a VAE:

```
   Text Prompt
        │
        V
 ┌─────────────┐
 │ CLIP Text   │  ───  Encodes the prompt into a sequence of 77 token embeddings
 │ Encoder     │        (each 768-dim).
 └──────┬──────┘
        │  context (1, 77, 768)
        V
 ┌─────────────┐
 │   U-Net     │  ───  Iteratively denoises a latent tensor (1, 4, 64, 64)
 │ (Diffusion) │        conditioned on CLIP embeddings + timestep.
 └──────┬──────┘
        │  denoised latent (1, 4, 64, 64)
        V
 ┌─────────────┐
 │ VAE Decoder │  ───  Upsamples the latent back to pixel space: (1, 3, 512, 512).
 └─────────────┘
```

The latent space is 8× smaller on each spatial side (512 -> 64), and uses 4
channels instead of 3 RGB channels, so the U-Net works on tensors that are
**48× smaller** than the pixel image. This is the key insight that makes
diffusion practical on consumer GPUs.

## What We Will Build

| Component | Purpose |
|||
| `SelfAttention` | `CrossAttention` | Shared attention building blocks |
| `VAE_Encoder` | Image -> latent (for img2img) |
| `VAE_Decoder` | Latent -> image |
| `CLIP` | Text prompt -> embeddings |
| `Diffusion` (U-Net) | Noise prediction conditioned on text + timestep |
| `DDPMSampler` | The denoising schedule |
| `generate()` | Ties everything together |

The pretrained weight conversion lives in `weights.py` (a mechanical key-mapping
from the official checkpoint to our naming convention — not relevant to
understanding the architecture).

## 0. Dependencies

In [ ]:
# setup up venv and install dependency
# !pip install -r requirement.txt

## 1. Attention Mechanisms

Attention is the workhorse of modern deep learning. SD 1.5 uses two flavours:

- **Self-Attention** — every position attends to every other position in the *same*
  sequence. Used inside the VAE (spatial self-attention over pixels) and inside the
  U-Net (spatial self-attention over latent patches) and CLIP (causal self-attention
  over token sequences).

- **Cross-Attention** — queries come from one modality (latent image features),
  while keys and values come from another (CLIP text embeddings). This is how the
  U-Net "reads" the text prompt.

Both implement multi-head attention:
$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import math
import numpy as np
from tqdm.notebook import tqdm


class SelfAttention(nn.Module):
    """
    Multi-head self-attention.

    Used in three places across SD 1.5:
      1. VAE encoder/decoder  — spatial self-attention over pixel features (1 head).
      2. CLIP text encoder     — causal self-attention over token embeddings (12 heads).
      3. U-Net attention blocks — self-attention over flattened latent features (8 heads).

    We merge Wq, Wk, Wv into a single linear projection (`in_proj`) for efficiency,
    then chunk the output into Q, K, V.
    """

    def __init__(self, n_heads, d_embed, in_proj_bias=True, out_proj_bias=True):
        super().__init__()
        # Single linear layer that produces Q, K, V concatenated: [Q | K | V]
        # Output dim = 3 * d_embed so we can chunk it into three equal parts
        self.in_proj = nn.Linear(d_embed, 3 * d_embed, bias=in_proj_bias)
        # Output projection Wo that recombines the multi-head outputs
        self.out_proj = nn.Linear(d_embed, d_embed, bias=out_proj_bias)
        self.n_heads = n_heads
        self.d_head = d_embed // n_heads  # Dimension per head

    def forward(self, x, causal_mask=False):
        # x: (Batch_Size, Seq_Len, Dim)
        # For images, Seq_Len = H*W (each pixel is a "token").
        # For CLIP, Seq_Len = 77 (text tokens).

        input_shape = x.shape
        batch_size, sequence_length, d_embed = input_shape

        # We will reshape to (Batch, Seq, Heads, Dim_per_Head) then transpose
        # to (Batch, Heads, Seq, Dim_per_Head) so each head attends independently.
        interim_shape = (batch_size, sequence_length, self.n_heads, self.d_head)

        # Project and split into Q, K, V
        # (B, Seq, Dim) -> (B, Seq, 3*Dim) -> 3 tensors of (B, Seq, Dim)
        q, k, v = self.in_proj(x).chunk(3, dim=-1)

        # Reshape for multi-head: (B, Seq, Dim) -> (B, Seq, H, D_h) -> (B, H, Seq, D_h)
        q = q.view(interim_shape).transpose(1, 2)
        k = k.view(interim_shape).transpose(1, 2)
        v = v.view(interim_shape).transpose(1, 2)

        # Scaled dot-product attention
        # (B, H, Seq, D_h) @ (B, H, D_h, Seq) -> (B, H, Seq, Seq)
        weight = q @ k.transpose(-1, -2)

        if causal_mask:
            # Used by CLIP: prevent tokens from attending to future positions.
            # Upper triangle (above diagonal) is set to -inf so softmax yields 0.
            mask = torch.ones_like(weight, dtype=torch.bool).triu(1)
            weight.masked_fill_(mask, -torch.inf)

        # Scale by sqrt(d_k) for stable gradients, then softmax over keys
        weight /= math.sqrt(self.d_head)
        weight = F.softmax(weight, dim=-1)

        # Weighted sum of values
        # (B, H, Seq, Seq) @ (B, H, Seq, D_h) -> (B, H, Seq, D_h)
        output = weight @ v

        # Concatenate heads: (B, H, Seq, D_h) -> (B, Seq, H, D_h) -> (B, Seq, Dim)
        output = output.transpose(1, 2).reshape(input_shape)

        # Final linear projection
        output = self.out_proj(output)

        return output


class CrossAttention(nn.Module):
    """
    Multi-head cross-attention.

    Queries come from the latent image features (d_embed),
    Keys and Values come from CLIP text embeddings (d_cross = 768).

    This is the mechanism that lets the U-Net "see" the text prompt:
      Q = latent features    (what the image is looking for)
      K = text embeddings    (what concepts are available)
      V = text embeddings    (the information to inject)

    Unlike self-attention, Q/K/V are projected from different sources,
    so we use separate linear layers instead of a merged in_proj.
    """

    def __init__(self, n_heads, d_embed, d_cross, in_proj_bias=True, out_proj_bias=True):
        super().__init__()
        self.q_proj   = nn.Linear(d_embed, d_embed, bias=in_proj_bias)  # Queries from latent
        self.k_proj   = nn.Linear(d_cross, d_embed, bias=in_proj_bias)  # Keys from text
        self.v_proj   = nn.Linear(d_cross, d_embed, bias=in_proj_bias)  # Values from text
        self.out_proj = nn.Linear(d_embed, d_embed, bias=out_proj_bias)
        self.n_heads = n_heads
        self.d_head = d_embed // n_heads

    def forward(self, x, y):
        # x (latent): (Batch, Seq_Q, Dim_Q)       e.g. (1, 4096, 320) for 64x64 latent
        # y (context): (Batch, Seq_KV, Dim_KV)    = (1, 77, 768)  from CLIP

        input_shape = x.shape
        batch_size, sequence_length, d_embed = input_shape
        interim_shape = (batch_size, -1, self.n_heads, self.d_head)

        # Project Q from latent features, K/V from text context
        q = self.q_proj(x)
        k = self.k_proj(y)  # Note: d_cross (768) -> d_embed via projection
        v = self.v_proj(y)

        # Reshape and transpose for multi-head attention
        q = q.view(interim_shape).transpose(1, 2)
        k = k.view(interim_shape).transpose(1, 2)
        v = v.view(interim_shape).transpose(1, 2)

        # Attention: each latent position attends to all 77 text tokens
        weight = q @ k.transpose(-1, -2)
        weight /= math.sqrt(self.d_head)
        weight = F.softmax(weight, dim=-1)

        output = weight @ v
        output = output.transpose(1, 2).contiguous()
        output = output.view(input_shape)
        output = self.out_proj(output)

        return output

## 2. VAE (Variational Autoencoder)

The VAE compresses images into a latent space and back. It was trained separately
(before the diffusion model) on a large image dataset using:
- **Reconstruction loss** (decoded image should match original)
- **KL-divergence loss** (latent distribution should stay close to standard normal)

### Why a Latent Space?
A 512×512×3 RGB image has ~786K values. The latent is 64×64×4 = ~16K values.
Running the U-Net on this smaller tensor is **~48× cheaper**, which is what makes
SD 1.5 practical.

### The 0.18215 Scaling Factor
The VAE encoder outputs latents whose empirical standard deviation is not exactly 1.0.
The factor `0.18215 ≈ 1 / sqrt(variance_of_latents)` rescales them to unit variance
so the diffusion model sees a well-normalized input. This value is specific to
SD 1.x/2.x (SDXL uses 0.13025).

### Building Blocks
Both encoder and decoder are built from:
- **Residual blocks** (GroupNorm -> SiLU -> Conv -> GroupNorm -> SiLU -> Conv + skip)
- **Attention blocks** (spatial self-attention so pixels can share global context)
- **Downsampling** (strided convolutions in encoder) / **Upsampling** (nearest + conv in decoder)

In [ ]:
class VAE_AttentionBlock(nn.Module):
    """
    Spatial self-attention block for the VAE.

    Flattens the spatial dimensions (H, W) into a sequence, applies self-attention
    (so every pixel can attend to every other pixel), then reshapes back.
    Uses a single head (n_heads=1) and a residual connection.
    """

    def __init__(self, channels):
        super().__init__()
        self.groupnorm = nn.GroupNorm(32, channels)
        self.attention = SelfAttention(1, channels)  # Single-head attention

    def forward(self, x):
        # x: (B, C, H, W)
        residue = x

        x = self.groupnorm(x)
        n, c, h, w = x.shape

        # Flatten spatial dims: (B, C, H, W) -> (B, C, H*W) -> (B, H*W, C)
        # Each pixel becomes a "token" with C features; sequence length = H*W
        x = x.view((n, c, h * w))
        x = x.transpose(-1, -2)

        # Self-attention WITHOUT causal mask (every pixel sees every other pixel)
        x = self.attention(x)

        # Reshape back: (B, H*W, C) -> (B, C, H*W) -> (B, C, H, W)
        x = x.transpose(-1, -2)
        x = x.view((n, c, h, w))

        # Residual connection
        x += residue
        return x


class VAE_ResidualBlock(nn.Module):
    """
    ResNet-style block: two conv layers with GroupNorm + SiLU activation,
    plus a skip connection. If in_channels != out_channels, a 1x1 conv
    adjusts the channel count on the skip path.

    GroupNorm (with 32 groups) is used instead of BatchNorm because:
    - It works with any batch size (even batch_size=1 during inference)
    - It normalizes within each sample independently
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.groupnorm_1 = nn.GroupNorm(32, in_channels)
        self.conv_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        self.groupnorm_2 = nn.GroupNorm(32, out_channels)
        self.conv_2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels == out_channels:
            self.residual_layer = nn.Identity()
        else:
            self.residual_layer = nn.Conv2d(in_channels, out_channels, kernel_size=1, padding=0)

    def forward(self, x):
        # x: (B, In_Channels, H, W)
        residue = x

        x = self.groupnorm_1(x)
        x = F.silu(x)  # SiLU (aka Swish): x * sigmoid(x)
        x = self.conv_1(x)

        x = self.groupnorm_2(x)
        x = F.silu(x)
        x = self.conv_2(x)

        return x + self.residual_layer(residue)

### 2a. VAE Encoder

The encoder takes a 512×512×3 RGB image and compresses it to a 64×64×4 latent.

It is a **variational** encoder: the final layers output 8 channels that are split
into a **mean** and **log-variance** (4 channels each). We then sample from this
Gaussian distribution using the reparameterization trick:

$$z = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

This is only used during **image-to-image** generation (text-to-image starts from
pure noise, bypassing the encoder).

Architecture overview:
```
Input (3, 512, 512)
  │
  ├─ Conv 3->128 + 2×ResBlock(128)          -> (128, 512, 512)
  ├─ Downsample (stride 2)                  -> (128, 256, 256)
  ├─ ResBlock(128->256) + ResBlock(256)     -> (256, 256, 256)
  ├─ Downsample (stride 2)                  -> (256, 128, 128)
  ├─ ResBlock(256->512) + ResBlock(512)     -> (512, 128, 128)
  ├─ Downsample (stride 2)                  -> (512, 64, 64)
  ├─ 3×ResBlock(512) + Attention(512)       -> (512, 64, 64)
  ├─ ResBlock(512) + GroupNorm + SiLU       -> (512, 64, 64)
  ├─ Conv 512->8 + Conv 8->8                -> (8, 64, 64)
  │
  └─ Split into mean(4) + logvar(4), sample -> (4, 64, 64)
```

In [ ]:
class VAE_Encoder(nn.Sequential):
    """
    Encodes a 512x512x3 image into a 64x64x4 latent representation.

    The encoder progressively downsamples the spatial resolution (512 -> 256 -> 128 -> 64)
    while increasing channels (3 -> 128 -> 256 -> 512 -> 8), then splits the output
    into mean and log-variance for the variational sampling.
    """

    def __init__(self):
        super().__init__(
            # Stage 1: 512x512, 128 channels 
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            VAE_ResidualBlock(128, 128),
            VAE_ResidualBlock(128, 128),

            # Downsample 512 -> 256 
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=0),

            # Stage 2: 256x256, 256 channels 
            VAE_ResidualBlock(128, 256),
            VAE_ResidualBlock(256, 256),

            # Downsample 256 -> 128 
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=0),

            # Stage 3: 128x128, 512 channels 
            VAE_ResidualBlock(256, 512),
            VAE_ResidualBlock(512, 512),

            # Downsample 128 -> 64 
            nn.Conv2d(512, 512, kernel_size=3, stride=2, padding=0),

            # Bottleneck: 64x64, 512 channels 
            VAE_ResidualBlock(512, 512),
            VAE_ResidualBlock(512, 512),
            VAE_ResidualBlock(512, 512),

            # Self-attention at the bottleneck lets distant pixels communicate
            VAE_AttentionBlock(512),

            VAE_ResidualBlock(512, 512),
            nn.GroupNorm(32, 512),
            nn.SiLU(),

            # Project to 8 channels: 4 for mean, 4 for log-variance
            nn.Conv2d(512, 8, kernel_size=3, padding=1),
            nn.Conv2d(8, 8, kernel_size=1, padding=0)
        )

    def forward(self, x: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, 512, 512) input image
        # noise: (B, 4, 64, 64) random noise for reparameterization trick

        for module in self:
            # Strided convolutions use asymmetric padding (pad right and bottom by 1)
            # to handle the odd-to-even dimension reduction correctly.
            if getattr(module, 'stride', None) == (2, 2):
                x = F.pad(x, (0, 1, 0, 1))  # (left, right, top, bottom)
            x = module(x)

        # Split 8 channels into mean and log_variance (4 channels each)
        # This is the "variational" part: we learn a distribution, not a point
        mean, log_variance = torch.chunk(x, 2, dim=1)

        # Clamp log_variance for numerical stability
        log_variance = torch.clamp(log_variance, -30, 20)
        variance = log_variance.exp()
        stdev = variance.sqrt()

        # Reparameterization trick: sample z = mean + stdev * noise
        # This lets gradients flow through the sampling operation
        x = mean + stdev * noise

        # Scale by the magic constant to normalize the latent distribution
        x *= 0.18215

        return x

### 2b. VAE Decoder

The decoder mirrors the encoder: it takes the 64×64×4 latent and progressively
upsamples it back to a 512×512×3 RGB image.

```
Input latent (4, 64, 64)
  │
  ├─ Conv 4->4 + Conv 4->512                -> (512, 64, 64)
  ├─ ResBlock(512) + Attention + 3×ResBlock -> (512, 64, 64)
  ├─ Upsample 2× + Conv                     -> (512, 128, 128)
  ├─ 3×ResBlock(512)                        -> (512, 128, 128)
  ├─ Upsample 2× + Conv                     -> (512, 256, 256)
  ├─ ResBlock(512->256) + 2×ResBlock(256)   -> (256, 256, 256)
  ├─ Upsample 2× + Conv                     -> (256, 512, 512)
  ├─ ResBlock(256->128) + 2×ResBlock(128)   -> (128, 512, 512)
  ├─ GroupNorm + SiLU + Conv 128->3         -> (3, 512, 512)
```

In [ ]:
class VAE_Decoder(nn.Sequential):
    """
    Decodes a 64x64x4 latent back to a 512x512x3 image.

    Mirrors the encoder architecture: starts with 512 channels at 64x64,
    progressively upsamples (64 -> 128 -> 256 -> 512) while reducing
    channels (512 -> 256 -> 128 -> 3).
    """

    def __init__(self):
        super().__init__(
            # Initial projection from 4 latent channels to 512
            nn.Conv2d(4, 4, kernel_size=1, padding=0),
            nn.Conv2d(4, 512, kernel_size=3, padding=1),

            # Bottleneck at 64x64 
            VAE_ResidualBlock(512, 512),
            VAE_AttentionBlock(512),  # Global spatial attention
            VAE_ResidualBlock(512, 512),
            VAE_ResidualBlock(512, 512),
            VAE_ResidualBlock(512, 512),
            VAE_ResidualBlock(512, 512),

            # Upsample 64 -> 128 
            nn.Upsample(scale_factor=2),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            VAE_ResidualBlock(512, 512),
            VAE_ResidualBlock(512, 512),
            VAE_ResidualBlock(512, 512),

            # Upsample 128 -> 256 
            nn.Upsample(scale_factor=2),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            VAE_ResidualBlock(512, 256),  # Channel reduction: 512 -> 256
            VAE_ResidualBlock(256, 256),
            VAE_ResidualBlock(256, 256),

            # Upsample 256 -> 512 
            nn.Upsample(scale_factor=2),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            VAE_ResidualBlock(256, 128),  # Channel reduction: 256 -> 128
            VAE_ResidualBlock(128, 128),
            VAE_ResidualBlock(128, 128),

            # Final output 
            nn.GroupNorm(32, 128),
            nn.SiLU(),
            nn.Conv2d(128, 3, kernel_size=3, padding=1),  # 128 -> 3 RGB channels
        )

    def forward(self, x):
        # x: (B, 4, 64, 64)

        # Undo the encoder's scaling
        x /= 0.18215

        for module in self:
            x = module(x)

        # (B, 3, 512, 512)
        return x

## 3. CLIP Text Encoder

SD 1.5 uses the **text encoder** from [CLIP ViT-L/14](https://openai.com/research/clip)
to convert the user's text prompt into a sequence of embedding vectors that the
U-Net can condition on via cross-attention.

### How it works
1. The prompt is **tokenized** into up to 77 tokens (padded if shorter).
2. Each token is mapped to a 768-dimensional embedding.
3. **Learned positional embeddings** are added (not sinusoidal).
4. The result passes through **12 transformer layers** (causal self-attention + FFN).
5. The output is a **(1, 77, 768)** tensor: one 768-dim vector per token position.

### Key design choices
- **Causal masking**: CLIP's text encoder uses causal (autoregressive) attention,
  even though we use the full sequence. This is inherited from CLIP's training.
- **QuickGELU**: `x * sigmoid(1.702 * x)` instead of the standard GELU. This is a
  fast approximation used in the original CLIP implementation.
- **Vocabulary size**: 49,408 BPE tokens.
- **We only use the text encoder** — CLIP's image encoder is not needed.

In [ ]:
class CLIPEmbedding(nn.Module):
    """Token + positional embedding for CLIP."""

    def __init__(self, n_vocab: int, n_embd: int, n_token: int):
        super().__init__()
        self.token_embedding = nn.Embedding(n_vocab, n_embd)
        # Learned positional embeddings (not sinusoidal like in the original Transformer)
        self.position_embedding = nn.Parameter(torch.zeros((n_token, n_embd)))

    def forward(self, tokens):
        # tokens: (B, 77) integer token IDs
        x = self.token_embedding(tokens)  # (B, 77, 768)
        x += self.position_embedding      # Broadcast add learned positions
        return x


class CLIPLayer(nn.Module):
    """
    One transformer layer of the CLIP text encoder.

    Pre-norm architecture: LayerNorm BEFORE attention/FFN (like GPT-2).
    Uses causal self-attention (each token can only see previous tokens).
    """

    def __init__(self, n_head: int, n_embd: int):
        super().__init__()
        self.layernorm_1 = nn.LayerNorm(n_embd)
        self.attention = SelfAttention(n_head, n_embd)
        self.layernorm_2 = nn.LayerNorm(n_embd)
        # FFN with 4x expansion (standard transformer ratio)
        self.linear_1 = nn.Linear(n_embd, 4 * n_embd)
        self.linear_2 = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        # Self-Attention block 
        residue = x
        x = self.layernorm_1(x)
        x = self.attention(x, causal_mask=True)  # Causal: inherited from CLIP training
        x += residue

        # Feed-Forward block 
        residue = x
        x = self.layernorm_2(x)
        x = self.linear_1(x)                        # (B, 77, 768) -> (B, 77, 3072)
        x = x * torch.sigmoid(1.702 * x)            # QuickGELU activation
        x = self.linear_2(x)                        # (B, 77, 3072) -> (B, 77, 768)
        x += residue

        return x


class CLIP(nn.Module):
    """
    CLIP ViT-L/14 Text Encoder.

    SD 1.5 specifics:
    - 12 transformer layers, 12 attention heads, 768-dim embeddings
    - Vocab: 49,408 BPE tokens, max sequence length: 77
    - ~123M parameters (text encoder only)
    """

    def __init__(self):
        super().__init__()
        self.embedding = CLIPEmbedding(49408, 768, 77)
        self.layers = nn.ModuleList([
            CLIPLayer(12, 768) for _ in range(12)
        ])
        self.layernorm = nn.LayerNorm(768)

    def forward(self, tokens: torch.LongTensor) -> torch.FloatTensor:
        tokens = tokens.type(torch.long)

        # (B, 77) -> (B, 77, 768)
        state = self.embedding(tokens)

        for layer in self.layers:
            state = layer(state)

        # Final layer norm
        output = self.layernorm(state)

        # Output: (B, 77, 768) - one embedding vector per token position
        return output

## 4. U-Net (Diffusion Model)

The U-Net is the core of the diffusion process. It takes a **noisy latent** and
predicts the **noise** that was added (so we can subtract it to denoise).

It is conditioned on two things:
1. **Text embeddings** from CLIP (injected via cross-attention)
2. **Timestep** (which noise level we're at, injected via addition after a learned MLP)

### Architecture: Encoder -> Bottleneck -> Decoder with Skip Connections

```
Encoder (downsampling)                 Decoder (upsampling)
──────────────────────                ────────────────────
320ch, 64x64  ────skip──────────>  concat -> 320ch, 64x64
320ch, 64x64  ────skip──────────>  concat -> 320ch, 64x64
320ch, 64x64  ────skip──────────>  concat -> 320ch, 64x64
320ch, 32x32  ────skip──────────>  concat -> 640ch, 32x32 ↑
640ch, 32x32  ────skip──────────>  concat -> 640ch, 32x32
640ch, 32x32  ────skip──────────>  concat -> 640ch, 32x32
640ch, 16x16  ────skip──────────>  concat -> 1280ch, 16x16 ↑
1280ch, 16x16 ────skip──────────>  concat -> 1280ch, 16x16
1280ch, 16x16 ────skip──────────>  concat -> 1280ch, 16x16
1280ch, 8x8   ────skip──────────>  concat -> 1280ch, 8x8 ↑
1280ch, 8x8   ────skip──────────>  concat -> 1280ch, 8x8
1280ch, 8x8   ────skip──────────>  concat -> 1280ch, 8x8
        \                      /
         ->-> Bottleneck (1280ch, 8x8) ←←
```

### Building Blocks

Each U-Net stage uses:
- **UNET_ResidualBlock**: Like VAE's, but with **timestep conditioning** — the timestep
  embedding is projected and added to the feature map.
- **UNET_AttentionBlock**: Contains self-attention (spatial), cross-attention (text),
  and a GeGLU feed-forward network.

### Time Embedding
The timestep (an integer 0–1000) is first converted to a sinusoidal positional
encoding (like in the original Transformer), then passed through a 2-layer MLP
to produce a 1280-dim vector.

In [ ]:
class TimeEmbedding(nn.Module):
    """
    Projects a sinusoidal timestep encoding (320-dim) to a richer representation (1280-dim).

    The sinusoidal encoding is computed externally (see `get_time_embedding()`).
    This MLP learns to transform it into a form the residual blocks can use.
    The 4x expansion (320 -> 1280) gives the network more capacity to represent
    the noise schedule.
    """

    def __init__(self, n_embd):
        super().__init__()
        self.linear_1 = nn.Linear(n_embd, 4 * n_embd)  # 320 -> 1280
        self.linear_2 = nn.Linear(4 * n_embd, 4 * n_embd)  # 1280 -> 1280

    def forward(self, x):
        # x: (1, 320) sinusoidal encoding
        x = self.linear_1(x)   # (1, 1280)
        x = F.silu(x)
        x = self.linear_2(x)   # (1, 1280)
        return x


class UNET_ResidualBlock(nn.Module):
    """
    Residual block with timestep conditioning.

    Similar to VAE's residual block, but the timestep embedding (1280-dim)
    is projected to out_channels and added to the feature map between the
    two conv layers. This tells the network "how noisy" the current input is.

    feature -> GroupNorm -> SiLU -> Conv -> (+time) -> GroupNorm -> SiLU -> Conv -> (+residual)
    """

    def __init__(self, in_channels, out_channels, n_time=1280):
        super().__init__()
        self.groupnorm_feature = nn.GroupNorm(32, in_channels)
        self.conv_feature = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.linear_time = nn.Linear(n_time, out_channels)  # Project time to channel dim

        self.groupnorm_merged = nn.GroupNorm(32, out_channels)
        self.conv_merged = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels == out_channels:
            self.residual_layer = nn.Identity()
        else:
            self.residual_layer = nn.Conv2d(in_channels, out_channels, kernel_size=1, padding=0)

    def forward(self, feature, time):
        # feature: (B, In_Ch, H, W)
        # time: (1, 1280)
        residue = feature

        feature = self.groupnorm_feature(feature)
        feature = F.silu(feature)
        feature = self.conv_feature(feature)

        # Project and add timestep embedding
        # (1, 1280) -> (1, Out_Ch) -> (1, Out_Ch, 1, 1) for broadcasting
        time = F.silu(time)
        time = self.linear_time(time)
        merged = feature + time.unsqueeze(-1).unsqueeze(-1)

        merged = self.groupnorm_merged(merged)
        merged = F.silu(merged)
        merged = self.conv_merged(merged)

        return merged + self.residual_layer(residue)


class UNET_AttentionBlock(nn.Module):
    """
    Transformer block inside the U-Net. Each block contains:

    1. Self-Attention  - spatial features attend to each other (image coherence)
    2. Cross-Attention - spatial features attend to CLIP text tokens (text guidance)
    3. GeGLU FFN       - non-linear feature transformation

    All three have pre-norm (LayerNorm) and residual connections.

    The input is a 2D feature map (B, C, H, W) which gets flattened to a sequence
    (B, H*W, C), processed by attention, then reshaped back.
    """

    def __init__(self, n_head: int, n_embd: int, d_context=768):
        super().__init__()
        channels = n_head * n_embd

        self.groupnorm = nn.GroupNorm(32, channels, eps=1e-6)
        self.conv_input = nn.Conv2d(channels, channels, kernel_size=1, padding=0)

        # Self-attention (spatial coherence)
        self.layernorm_1 = nn.LayerNorm(channels)
        self.attention_1 = SelfAttention(n_head, channels, in_proj_bias=False)

        # Cross-attention (text conditioning)
        self.layernorm_2 = nn.LayerNorm(channels)
        self.attention_2 = CrossAttention(n_head, channels, d_context, in_proj_bias=False)

        # GeGLU Feed-Forward Network
        self.layernorm_3 = nn.LayerNorm(channels)
        self.linear_geglu_1 = nn.Linear(channels, 4 * channels * 2)  # 2x for the gate
        self.linear_geglu_2 = nn.Linear(4 * channels, channels)

        self.conv_output = nn.Conv2d(channels, channels, kernel_size=1, padding=0)

    def forward(self, x, context):
        # x: (B, C, H, W) - image features
        # context: (B, 77, 768) - CLIP text embeddings
        residue_long = x

        x = self.groupnorm(x)
        x = self.conv_input(x)

        n, c, h, w = x.shape

        # Flatten spatial dims to sequence: (B, C, H, W) -> (B, H*W, C)
        x = x.view((n, c, h * w))
        x = x.transpose(-1, -2)

        # 1. Self-Attention + residual
        residue_short = x
        x = self.layernorm_1(x)
        x = self.attention_1(x)
        x += residue_short

        # 2. Cross-Attention + residual (this is where text guides the image)
        residue_short = x
        x = self.layernorm_2(x)
        x = self.attention_2(x, context)
        x += residue_short

        # 3. GeGLU Feed-Forward + residual
        # GeGLU: split the linear output into two halves, one is gated by GELU
        # This is more expressive than a simple FFN
        residue_short = x
        x = self.layernorm_3(x)
        x, gate = self.linear_geglu_1(x).chunk(2, dim=-1)
        x = x * F.gelu(gate)  # Element-wise gating
        x = self.linear_geglu_2(x)
        x += residue_short

        # Reshape back to 2D feature map: (B, H*W, C) -> (B, C, H, W)
        x = x.transpose(-1, -2)
        x = x.view((n, c, h, w))

        # Long skip connection around the entire block
        return self.conv_output(x) + residue_long


class Upsample(nn.Module):
    """2x nearest-neighbor upsampling followed by a 3x3 conv."""

    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        return self.conv(x)


class SwitchSequential(nn.Sequential):
    """
    A Sequential container that routes `context` and `time` to the right layers:
    - UNET_AttentionBlock gets (x, context)
    - UNET_ResidualBlock gets (x, time)
    - Everything else (Conv2d, Upsample) just gets (x)
    """

    def forward(self, x, context=None, time=None):
        for layer in self:
            if isinstance(layer, UNET_AttentionBlock):
                x = layer(x, context)
            elif isinstance(layer, UNET_ResidualBlock):
                x = layer(x, time)
            else:
                x = layer(x)
        return x

### 4a. The Full U-Net

The U-Net has three main parts:

1. **Encoder** (12 blocks): Progressively downsamples 64->32->16->8 while increasing
   channels 320->640->1280. Uses strided convolutions for downsampling.

2. **Bottleneck** (1 block): ResBlock + Attention + ResBlock at the deepest level.

3. **Decoder** (12 blocks): Mirrors the encoder, upsampling back. Each decoder block
   receives the **concatenated** encoder features via skip connections (hence 2x
   input channels: e.g., 1280 from decoder + 1280 from encoder = 2560 input).

The skip connections are crucial: without them the network would lose fine-grained
spatial information during downsampling.

In [ ]:
class UNET(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoders = nn.ModuleList([
            # Level 0: 64x64, 320 channels
            # Initial projection from 4 latent channels to 320
            SwitchSequential(nn.Conv2d(4, 320, kernel_size=3, padding=1)),

            # Two (ResBlock + Attention) blocks at 64x64
            SwitchSequential(UNET_ResidualBlock(320, 320), UNET_AttentionBlock(8, 40)),
            SwitchSequential(UNET_ResidualBlock(320, 320), UNET_AttentionBlock(8, 40)),

            # Downsample: 64x64 -> 32x32
            SwitchSequential(nn.Conv2d(320, 320, kernel_size=3, stride=2, padding=1)),

            # Level 1: 32x32, 640 channels
            SwitchSequential(UNET_ResidualBlock(320, 640), UNET_AttentionBlock(8, 80)),
            SwitchSequential(UNET_ResidualBlock(640, 640), UNET_AttentionBlock(8, 80)),

            # Downsample: 32x32 -> 16x16
            SwitchSequential(nn.Conv2d(640, 640, kernel_size=3, stride=2, padding=1)),

            # Level 2: 16x16, 1280 channels
            SwitchSequential(UNET_ResidualBlock(640, 1280), UNET_AttentionBlock(8, 160)),
            SwitchSequential(UNET_ResidualBlock(1280, 1280), UNET_AttentionBlock(8, 160)),

            # Downsample: 16x16 -> 8x8
            SwitchSequential(nn.Conv2d(1280, 1280, kernel_size=3, stride=2, padding=1)),

            # Level 3: 8x8, 1280 channels (no attention at deepest level)
            SwitchSequential(UNET_ResidualBlock(1280, 1280)),
            SwitchSequential(UNET_ResidualBlock(1280, 1280)),
        ])

        self.bottleneck = SwitchSequential(
            # Deepest point: 8x8, 1280 channels
            UNET_ResidualBlock(1280, 1280),
            UNET_AttentionBlock(8, 160),
            UNET_ResidualBlock(1280, 1280),
        )

        self.decoders = nn.ModuleList([
            # Level 3 (decoder): 8x8
            # Input channels are 2x because of skip connection concatenation
            SwitchSequential(UNET_ResidualBlock(2560, 1280)),
            SwitchSequential(UNET_ResidualBlock(2560, 1280)),
            SwitchSequential(UNET_ResidualBlock(2560, 1280), Upsample(1280)),  # 8x8 -> 16x16

            # Level 2 (decoder): 16x16
            SwitchSequential(UNET_ResidualBlock(2560, 1280), UNET_AttentionBlock(8, 160)),
            SwitchSequential(UNET_ResidualBlock(2560, 1280), UNET_AttentionBlock(8, 160)),
            SwitchSequential(UNET_ResidualBlock(1920, 1280), UNET_AttentionBlock(8, 160), Upsample(1280)),  # 16x16 -> 32x32

            # Level 1 (decoder): 32x32
            SwitchSequential(UNET_ResidualBlock(1920, 640), UNET_AttentionBlock(8, 80)),
            SwitchSequential(UNET_ResidualBlock(1280, 640), UNET_AttentionBlock(8, 80)),
            SwitchSequential(UNET_ResidualBlock(960, 640), UNET_AttentionBlock(8, 80), Upsample(640)),  # 32x32 -> 64x64

            # Level 0 (decoder): 64x64
            SwitchSequential(UNET_ResidualBlock(960, 320), UNET_AttentionBlock(8, 40)),
            SwitchSequential(UNET_ResidualBlock(640, 320), UNET_AttentionBlock(8, 40)),
            SwitchSequential(UNET_ResidualBlock(640, 320), UNET_AttentionBlock(8, 40)),
        ])

    def forward(self, x, context, time):
        # x: (B, 4, 64, 64) noisy latent
        # context: (B, 77, 768) CLIP text embeddings
        # time: (1, 1280) processed timestep embedding

        # Encoder: save outputs for skip connections
        skip_connections = []
        for layers in self.encoders:
            x = layers(x, context, time)
            skip_connections.append(x)

        # Bottleneck
        x = self.bottleneck(x, context, time)

        # Decoder: concatenate with skip connections (LIFO order)
        for layers in self.decoders:
            x = torch.cat((x, skip_connections.pop()), dim=1)
            x = layers(x, context, time)

        return x


class UNET_OutputLayer(nn.Module):
    """Final layer: projects 320 channels back to 4 latent channels."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.groupnorm = nn.GroupNorm(32, in_channels)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.groupnorm(x)
        x = F.silu(x)
        x = self.conv(x)
        return x


class Diffusion(nn.Module):
    """
    The complete diffusion model: TimeEmbedding + UNET + OutputLayer.

    Takes a noisy latent, the CLIP context, and a timestep encoding.
    Returns the predicted noise (same shape as the input latent).
    """

    def __init__(self):
        super().__init__()
        self.time_embedding = TimeEmbedding(320)
        self.unet = UNET()
        self.final = UNET_OutputLayer(320, 4)

    def forward(self, latent, context, time):
        # latent: (B, 4, 64, 64) - noisy latent
        # context: (B, 77, 768) - text embeddings
        # time: (1, 320) - sinusoidal timestep encoding

        # Project timestep: (1, 320) -> (1, 1280)
        time = self.time_embedding(time)

        # U-Net forward pass: (B, 4, 64, 64) -> (B, 320, 64, 64)
        output = self.unet(latent, context, time)

        # Project back to latent channels: (B, 320, 64, 64) -> (B, 4, 64, 64)
        output = self.final(output)

        return output  # Predicted noise

## 5. DDPM Sampler

The sampler implements the **denoising** process. During training, the model learns
to predict noise added at various levels. During inference, we iteratively subtract
the predicted noise to recover a clean image.

### The Noise Schedule
DDPM (Denoising Diffusion Probabilistic Models) defines a **forward process** that
gradually adds Gaussian noise to an image over T=1000 timesteps:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar\alpha_t} \, x_0, \; (1 - \bar\alpha_t) I)$$

where $\bar\alpha_t = \prod_{s=1}^{t} (1 - \beta_s)$ and $\beta_t$ increases from
0.00085 to 0.012 (SD 1.5 uses a **scaled linear** schedule: linearly spaced in
$\sqrt{\beta}$ space).

### The Reverse Process
At each step, the model predicts the noise $\epsilon_\theta(x_t, t)$, and we compute:

$$x_{t-1} = \frac{\sqrt{\bar\alpha_{t-1}} \beta_t}{1 - \bar\alpha_t} \hat{x}_0
           + \frac{\sqrt{\alpha_t}(1 - \bar\alpha_{t-1})}{1 - \bar\alpha_t} x_t
           + \sigma_t z$$

where $\hat{x}_0$ is the predicted clean image and $z \sim \mathcal{N}(0, I)$.

### Inference Speedup
Training uses T=1000 steps, but inference only uses ~50 evenly-spaced steps.

In [ ]:
class DDPMSampler:
    """
    DDPM (Denoising Diffusion Probabilistic Models) sampler.

    Implements the reverse diffusion process: given the model's noise predictions,
    iteratively denoise from x_T (pure noise) back to x_0 (clean latent).

    Reference: https://arxiv.org/pdf/2006.11239.pdf
    """

    def __init__(self, generator: torch.Generator, num_training_steps=1000,
                 beta_start: float = 0.00085, beta_end: float = 0.0120):
        # Build the noise schedule 
        # SD 1.5 uses a "scaled linear" schedule: linear in sqrt(beta) space
        # This gives a gentler noise increase at the start
        self.betas = torch.linspace(
            beta_start ** 0.5, beta_end ** 0.5, num_training_steps, dtype=torch.float32
        ) ** 2

        # alpha_t = 1 - beta_t
        self.alphas = 1.0 - self.betas

        # alpha_bar_t = cumulative product of alphas (key quantity for the forward process)
        # alpha_bar_t tells us the total noise level at step t
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.one = torch.tensor(1.0)

        self.generator = generator
        self.num_train_timesteps = num_training_steps
        self.timesteps = torch.from_numpy(np.arange(0, num_training_steps)[::-1].copy())

    def set_inference_timesteps(self, num_inference_steps=50):
        """
        Set up evenly-spaced timesteps for inference.
        E.g., with 50 steps and 1000 training steps, we use timesteps:
        [980, 960, 940, ..., 20, 0]
        """
        self.num_inference_steps = num_inference_steps
        step_ratio = self.num_train_timesteps // self.num_inference_steps
        timesteps = (np.arange(0, num_inference_steps) * step_ratio).round()[::-1].copy().astype(np.int64)
        self.timesteps = torch.from_numpy(timesteps)

    def _get_previous_timestep(self, timestep: int) -> int:
        prev_t = timestep - self.num_train_timesteps // self.num_inference_steps
        return prev_t

    def _get_variance(self, timestep: int) -> torch.Tensor:
        """Compute the variance for the posterior q(x_{t-1} | x_t, x_0)."""
        prev_t = self._get_previous_timestep(timestep)

        alpha_prod_t = self.alphas_cumprod[timestep]
        alpha_prod_t_prev = self.alphas_cumprod[prev_t] if prev_t >= 0 else self.one
        current_beta_t = 1 - alpha_prod_t / alpha_prod_t_prev

        # Formula (7) from the DDPM paper
        variance = (1 - alpha_prod_t_prev) / (1 - alpha_prod_t) * current_beta_t
        variance = torch.clamp(variance, min=1e-20)
        return variance

    def set_strength(self, strength=1):
        """
        For image-to-image: controls how much the output deviates from the input.
        - strength=1.0: full denoising (like text-to-image)
        - strength=0.5: start halfway through the schedule (less change)
        - strength=0.0: no change at all
        """
        start_step = self.num_inference_steps - int(self.num_inference_steps * strength)
        self.timesteps = self.timesteps[start_step:]
        self.start_step = start_step

    def step(self, timestep: int, latents: torch.Tensor, model_output: torch.Tensor):
        """
        One denoising step: given the current noisy latent and the model's
        noise prediction, compute the slightly-less-noisy latent x_{t-1}.
        """
        t = timestep
        prev_t = self._get_previous_timestep(t)

        # 1. Look up noise schedule values
        alpha_prod_t = self.alphas_cumprod[t]
        alpha_prod_t_prev = self.alphas_cumprod[prev_t] if prev_t >= 0 else self.one
        beta_prod_t = 1 - alpha_prod_t
        beta_prod_t_prev = 1 - alpha_prod_t_prev
        current_alpha_t = alpha_prod_t / alpha_prod_t_prev
        current_beta_t = 1 - current_alpha_t

        # 2. Predict the clean image x_0 from the noise prediction
        # From equation (15): x_0 = (x_t - sqrt(1-alpha_bar_t) * eps) / sqrt(alpha_bar_t)
        pred_original_sample = (latents - beta_prod_t ** 0.5 * model_output) / alpha_prod_t ** 0.5

        # 3. Compute the mean of q(x_{t-1} | x_t, x_0) from formula (7)
        pred_original_sample_coeff = (alpha_prod_t_prev ** 0.5 * current_beta_t) / beta_prod_t
        current_sample_coeff = current_alpha_t ** 0.5 * beta_prod_t_prev / beta_prod_t
        pred_prev_sample = pred_original_sample_coeff * pred_original_sample + current_sample_coeff * latents

        # 4. Add noise (except at the final step t=0)
        variance = 0
        if t > 0:
            device = model_output.device
            noise = torch.randn(
                model_output.shape, generator=self.generator,
                device=device, dtype=model_output.dtype
            )
            variance = (self._get_variance(t) ** 0.5) * noise

        # x_{t-1} = mean + sigma * z
        pred_prev_sample = pred_prev_sample + variance
        return pred_prev_sample

    def add_noise(self, original_samples: torch.FloatTensor,
                  timesteps: torch.IntTensor) -> torch.FloatTensor:
        """
        Forward process: add noise to clean samples at a given timestep.
        Used in image-to-image to create a noisy starting latent.

        q(x_t | x_0) = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
        """
        alphas_cumprod = self.alphas_cumprod.to(
            device=original_samples.device, dtype=original_samples.dtype
        )
        timesteps = timesteps.to(original_samples.device)

        sqrt_alpha_prod = alphas_cumprod[timesteps] ** 0.5
        sqrt_alpha_prod = sqrt_alpha_prod.flatten()
        while len(sqrt_alpha_prod.shape) < len(original_samples.shape):
            sqrt_alpha_prod = sqrt_alpha_prod.unsqueeze(-1)

        sqrt_one_minus_alpha_prod = (1 - alphas_cumprod[timesteps]) ** 0.5
        sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.flatten()
        while len(sqrt_one_minus_alpha_prod.shape) < len(original_samples.shape):
            sqrt_one_minus_alpha_prod = sqrt_one_minus_alpha_prod.unsqueeze(-1)

        noise = torch.randn(
            original_samples.shape, generator=self.generator,
            device=original_samples.device, dtype=original_samples.dtype
        )
        noisy_samples = sqrt_alpha_prod * original_samples + sqrt_one_minus_alpha_prod * noise
        return noisy_samples

## 6. Generation Pipeline

The `generate()` function ties all components together. Here is the complete flow:

### Text-to-Image Flow
```
1. Tokenize prompt with CLIP tokenizer (-> 77 tokens)
2. Encode tokens with CLIP (-> (1, 77, 768) text embeddings)
3. If using CFG: also encode the negative/empty prompt
4. Start with pure random noise in latent space (1, 4, 64, 64)
5. For each timestep (from noisy to clean):
   a. Compute sinusoidal time embedding
   b. Run U-Net to predict noise (with both prompts if CFG)
   c. Apply Classifier-Free Guidance: amplify the difference
   d. Use DDPM to take one denoising step
6. Decode the final latent with VAE decoder -> RGB image
```

### Classifier-Free Guidance (CFG)
CFG makes the model follow the prompt more closely by amplifying the difference
between conditional and unconditional predictions:

$$\epsilon_{\text{guided}} = \epsilon_{\text{uncond}} + s \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$$

where $s$ is the `cfg_scale` (typically 7-8). Higher values = more prompt adherence
but potentially less diversity/quality.

In [ ]:
WIDTH = 512
HEIGHT = 512
LATENTS_WIDTH = WIDTH // 8    # 64
LATENTS_HEIGHT = HEIGHT // 8  # 64


def generate(
    prompt,
    uncond_prompt=None,     # Negative prompt (empty string for none)
    input_image=None,       # PIL Image for img2img (None for txt2img)
    strength=0.8,           # img2img: how much to change (0=none, 1=full)
    do_cfg=True,            # Whether to use Classifier-Free Guidance
    cfg_scale=7.5,          # CFG strength (1=no guidance, 7-8=typical, 14=max)
    sampler_name="ddpm",
    n_inference_steps=50,
    models={},
    seed=None,
    device=None,
    idle_device=None,       # Move models here when not in use (saves VRAM)
    tokenizer=None,
):
    with torch.no_grad():
        if not 0 < strength <= 1:
            raise ValueError("strength must be between 0 and 1")

        if idle_device:
            to_idle = lambda x: x.to(idle_device)
        else:
            to_idle = lambda x: x

        # Step 1: Initialize RNG 
        generator = torch.Generator(device=device)
        if seed is None:
            generator.seed()
        else:
            generator.manual_seed(seed)

        # Step 2: Encode text prompt with CLIP 
        clip = models["clip"]
        clip.to(device)

        if do_cfg:
            # Encode both the prompt and the negative/empty prompt
            # We batch them together for efficiency: [conditional, unconditional]
            cond_tokens = tokenizer(
                [prompt], padding="max_length", max_length=77, truncation=True
            ).input_ids
            cond_tokens = torch.tensor(cond_tokens, dtype=torch.long, device=device)
            cond_context = clip(cond_tokens)  # (1, 77, 768)

            uncond_tokens = tokenizer(
                [uncond_prompt], padding="max_length", max_length=77, truncation=True
            ).input_ids
            uncond_tokens = torch.tensor(uncond_tokens, dtype=torch.long, device=device)
            uncond_context = clip(uncond_tokens)  # (1, 77, 768)

            # Stack: (2, 77, 768) - process both in one forward pass
            context = torch.cat([cond_context, uncond_context])
        else:
            tokens = tokenizer(
                [prompt], padding="max_length", max_length=77, truncation=True
            ).input_ids
            tokens = torch.tensor(tokens, dtype=torch.long, device=device)
            context = clip(tokens)  # (1, 77, 768)

        to_idle(clip)  # Free VRAM

        # Step 3: Initialize sampler 
        if sampler_name == "ddpm":
            sampler = DDPMSampler(generator)
            sampler.set_inference_timesteps(n_inference_steps)
        else:
            raise ValueError(f"Unknown sampler: {sampler_name}")

        latents_shape = (1, 4, LATENTS_HEIGHT, LATENTS_WIDTH)

        # Step 4: Create initial latent 
        if input_image:
            # Image-to-Image: encode the input image to latent space, then add noise
            encoder = models["encoder"]
            encoder.to(device)

            input_image_tensor = input_image.resize((WIDTH, HEIGHT))
            input_image_tensor = np.array(input_image_tensor)
            input_image_tensor = torch.tensor(input_image_tensor, dtype=torch.float32, device=device)
            # Rescale from [0, 255] to [-1, 1] (the range the VAE expects)
            input_image_tensor = rescale(input_image_tensor, (0, 255), (-1, 1))
            # (H, W, C) -> (1, C, H, W)
            input_image_tensor = input_image_tensor.unsqueeze(0).permute(0, 3, 1, 2)

            # Encode to latent space
            encoder_noise = torch.randn(latents_shape, generator=generator, device=device)
            latents = encoder(input_image_tensor, encoder_noise)

            # Add noise according to the strength parameter
            sampler.set_strength(strength=strength)
            latents = sampler.add_noise(latents, sampler.timesteps[0])
            to_idle(encoder)
        else:
            # Text-to-Image: start from pure random noise
            latents = torch.randn(latents_shape, generator=generator, device=device)

        # Step 5: Iterative denoising loop 
        diffusion = models["diffusion"]
        diffusion.to(device)

        timesteps = tqdm(sampler.timesteps)
        for i, timestep in enumerate(timesteps):
            # Compute sinusoidal time embedding: scalar -> (1, 320)
            time_embedding = get_time_embedding(timestep).to(device)

            model_input = latents

            if do_cfg:
                # Duplicate the latent for conditional + unconditional pass
                model_input = model_input.repeat(2, 1, 1, 1)  # (2, 4, 64, 64)

            # U-Net predicts the noise
            model_output = diffusion(model_input, context, time_embedding)

            if do_cfg:
                # Split predictions and apply CFG formula
                output_cond, output_uncond = model_output.chunk(2)
                model_output = cfg_scale * (output_cond - output_uncond) + output_uncond

            # One denoising step
            latents = sampler.step(timestep, latents, model_output)

        to_idle(diffusion)

        # Step 6: Decode latent to image 
        decoder = models["decoder"]
        decoder.to(device)
        images = decoder(latents)  # (1, 3, 512, 512)
        to_idle(decoder)

        # Post-process: rescale from [-1, 1] to [0, 255]
        images = rescale(images, (-1, 1), (0, 255), clamp=True)
        # (B, C, H, W) -> (B, H, W, C) for PIL/numpy
        images = images.permute(0, 2, 3, 1)
        images = images.to("cpu", torch.uint8).numpy()
        return images[0]


def rescale(x, old_range, new_range, clamp=False):
    """Linearly rescale tensor values from old_range to new_range."""
    old_min, old_max = old_range
    new_min, new_max = new_range
    x -= old_min
    x *= (new_max - new_min) / (old_max - old_min)
    x += new_min
    if clamp:
        x = x.clamp(new_min, new_max)
    return x


def get_time_embedding(timestep):
    """
    Sinusoidal positional encoding for the timestep, identical to the one
    used in the original Transformer paper ("Attention Is All You Need").

    Maps a single integer timestep to a 320-dim vector of sines and cosines
    at geometrically-spaced frequencies.
    """
    # 160 frequencies, geometrically spaced from 1 to 1/10000
    freqs = torch.pow(10000, -torch.arange(start=0, end=160, dtype=torch.float32) / 160)
    # Outer product: (1,) x (160,) -> (1, 160)
    x = torch.tensor([timestep], dtype=torch.float32)[:, None] * freqs[None]
    # Concat cos and sin: (1, 320)
    return torch.cat([torch.cos(x), torch.sin(x)], dim=-1)

## 7. Loading Pretrained Weights

The weight conversion logic lives in `weights.py`. It loads the official SD 1.5
checkpoint (`.ckpt` format from CompVis/Stability AI) and maps ~900 parameter names
from the original naming convention to our architecture.

This is purely mechanical plumbing — for example:
- `model.diffusion_model.input_blocks.1.0.in_layers.0.weight` -> `unet.encoders.1.0.groupnorm_feature.weight`
- `first_stage_model.encoder.down.0.block.0.norm1.weight` -> `encoder.1.groupnorm_1.weight`

Not relevant to understanding SD, so we keep it separate.

In [ ]:
from weights import load_from_standard_weights


def preload_models_from_standard_weights(ckpt_path, device):
    """
    Load all four model components from a single .ckpt file.

    Returns a dict with keys: 'clip', 'encoder', 'decoder', 'diffusion'
    """
    state_dict = load_from_standard_weights(ckpt_path, device)

    encoder = VAE_Encoder().to(device)
    encoder.load_state_dict(state_dict['encoder'], strict=True)

    decoder = VAE_Decoder().to(device)
    decoder.load_state_dict(state_dict['decoder'], strict=True)

    diffusion = Diffusion().to(device)
    diffusion.load_state_dict(state_dict['diffusion'], strict=True)

    clip = CLIP().to(device)
    clip.load_state_dict(state_dict['clip'], strict=True)

    return {
        'clip': clip,
        'encoder': encoder,
        'decoder': decoder,
        'diffusion': diffusion,
    }

## 8. Download Model & Run Inference

Now let's put it all together! We'll:
1. Download the SD 1.5 checkpoint from HuggingFace (~4GB)
2. Load the CLIP tokenizer
3. Run **text-to-image** generation
4. Run **image-to-image** generation

In [ ]:
from PIL import Image
from pathlib import Path
from transformers import CLIPTokenizer
import json

In [ ]:
from huggingface_hub import hf_hub_download

# Download the pruned (EMA-only) checkpoint - smaller but works great for inference
model_file = hf_hub_download(
    repo_id="stable-diffusion-v1-5/stable-diffusion-v1-5",
    filename="v1-5-pruned-emaonly.ckpt"
)

In [ ]:
# Download a sample image for image-to-image demo
!wget -q https://upload.wikimedia.org/wikipedia/commons/thumb/7/74/A-Cat.jpg/1280px-A-Cat.jpg

In [ ]:
# Device setup 
DEVICE = "cpu"
ALLOW_CUDA = True
ALLOW_MPS = False

if torch.cuda.is_available() and ALLOW_CUDA:
    DEVICE = "cuda"
elif (torch.has_mps or torch.backends.mps.is_available()) and ALLOW_MPS:
    DEVICE = "mps"
print(f"Using device: {DEVICE}")

# Load CLIP tokenizer (BPE vocabulary) 
tokenizer = CLIPTokenizer.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    subfolder="tokenizer"
)

# Load all models from the checkpoint 
models = preload_models_from_standard_weights(model_file, DEVICE)

### 8a. Text-to-Image

In [ ]:
# Text-to-Image 
prompt = "A cat with sunglasses, wearing comfy hat, highly detailed, ultra sharp, cinematic, 100mm lens, 8k resolution."
uncond_prompt = ""   # Empty negative prompt
do_cfg = True
cfg_scale = 8        # How strongly to follow the prompt (1=ignore, 14=maximum)

output_image = generate(
    prompt=prompt,
    uncond_prompt=uncond_prompt,
    input_image=None,       # No input image = text-to-image
    strength=1,             # Ignored for text-to-image
    do_cfg=do_cfg,
    cfg_scale=cfg_scale,
    sampler_name="ddpm",
    n_inference_steps=50,
    seed=42,
    models=models,
    device=DEVICE,
    idle_device="cuda",
    tokenizer=tokenizer,
)

output_pil = Image.fromarray(output_image)
output_pil.save("output_txt2img.png")
output_pil

### 8b. Image-to-Image

Image-to-image starts by encoding the input image to latent space, adding noise
at a level determined by `strength`, then denoising with the text prompt.
The result preserves the overall structure of the input while transforming it
according to the prompt.

In [ ]:
# Image-to-Image 
image_path = "1280px-A-Cat.jpg"
input_image = Image.open(image_path).convert("RGB")
input_image = input_image.resize((512, 512))

prompt = "A cat wearing sunglasses, highly detailed, ultra sharp, cinematic, 100mm lens, 8k resolution."
uncond_prompt = ""
strength = 0.8  # 0.8 = significant transformation while keeping structure

output_image = generate(
    prompt=prompt,
    uncond_prompt=uncond_prompt,
    input_image=input_image,
    strength=strength,
    do_cfg=True,
    cfg_scale=8,
    sampler_name="ddpm",
    n_inference_steps=50,
    seed=42,
    models=models,
    device=DEVICE,
    idle_device="cuda",
    tokenizer=tokenizer,
)

output_pil = Image.fromarray(output_image)
output_pil.save("output_img2img.png")

print("Input image:")
display(input_image)
print("\nOutput image:")
display(output_pil)